#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [5]:
# requirements

from pyspark.sql import SparkSession
from functools import reduce
from pyspark.sql.functions import col, when, lit, regexp_extract, upper, udf
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
import re


In [6]:
# txn_type pattern 配置字典（顺序很重要，跟 SQL 的 case when 一样）

bank_txn_type_pattern_rules = {
    "BCA": [
        # (patterns, cr_db, txn_type)
        # === CR ===
        (["KR OTOMATIS%QR%", "TRSF E-BANKING CR%FTQRS%"], "CR", "QR Transfer"),
        (["TRSF E-BANKING CR%"], "CR", "Bank Transfer"),
        (["BI-FAST CR%"], "CR", "BIF Transfer"),
        (["SETORAN%"], "CR", "Deposit"),
        (["CR KOREKSI%"], "CR", "Correction"),
        (["BUNGA%"], "CR", "Interest Income"),
        (["KR OTOMATIS%"], "CR", "Automatic"),
        (["SWITCHING%"], "CR", "Switching"),
        (["KR INTERCHANGE%"], "CR", "Interchange"),
        # === DB ===
        (["TRSF E-BANKING DB%"], "DB", "Bank Transfer"),
        (["BI-FAST DB%"], "DB", "BIF Transfer"),
        (["TRANSAKSI DEBIT%TGL%"], "DB", "QR Payment"),
        (["DB OTOMATIS%", "DEBET OTOMATIS%"], "DB", "Automatic Debit"),
        (["SWITCHING%"], "DB", "Switching"),
        ([
            "DB INTERCHANGE%",
            "DR KOREKSI%",
            "KR OTOMATIS KOR DEBIT%",
            "KR DEBIT DOMESTIK KOR DEBIT DOM%"
        ], "DB", "Correction"),
        (["KARTU DEBIT%"], "DB", "Card Payment"),
        (["%KARTU KREDIT%", "PEMBAYARAN%"], "DB", "Repayment"),
        ([
            "BIAYA ADM%",
            "BYR VIA E-BANKING%",
            "PAJAK BUNGA%",
            "BIAYA KARTU ATM%",
            "DB DEBIT DOMESTIK%",
            "BA JASA E-BANKING%",
            "FLAZZ%",
            "TARIKAN ATM%",
            "TARIKAN TUNAI%",
            "TARIKANE%",
            "TARIKAN PEMINDAHAN%",
            "ND-%",
            "TOLAKAN KLIRING%",
            "COLL DB%"
        ], "DB", "Other Account Related"),
    ],
    "MANDIRI": [],
    "BNI": [],
    "BRI": [],
    "SEABANK": []
}

def build_txn_type_column(df, rules, result_col_name="txn_type"):
    condition = None
    for bank, pattern_list in rules.items():
        for patterns, cr_db, txn_type in pattern_list:
            for pattern in patterns:
                regex = re.escape(pattern).replace(r"%", ".*")
                match_expr = (
                    (col("bank") == bank) &
                    (col("cr_db") == cr_db) &
                    col("description").rlike(regex)
                )
                condition = when(match_expr, txn_type) if condition is None else condition.when(match_expr, txn_type)

    condition = condition.otherwise(lit(None))
    return df.withColumn(result_col_name, condition)


In [7]:
# comp_or_indiv pattern 配置字典

bank_comp_or_indiv_pattern_rules = {
    "BCA": [
        # (patterns, cr_db, txn_type)
        # === CR ===
        (["TRSF E-BANKING CR SMEMFTS%", 
          "TRSF E-BANKING CR%FTFVA/WS95181%", 
          "TRSF E-BANKING CR%WS95051%", 
          "TRSF E-BANKING CR%PYBCA%", 
          "TRSF E-BANKING CR%ACBCA/WS95082%"], 
         "CR", "company"),
        
        (["TRSF E-BANKING CR% WSID%", 
          "TRSF E-BANKING CR %/0000%", 
          "TRSF E-BANKING CR%FTQRS%", 
          "TRSF E-BANKING CR%FTSCY/WS95031%", 
          "TRSF E-BANKING CR%FTSCY/WS95271%", 
          "TRSF E-BANKING CR%FTMCA/WS95271%", 
          "TRSF E-BANKING CR%FTSCY/WS95011%", 
          "TRSF E-BANKING CR%FTSCY/WS95261%"], 
         "CR", "individual"),

        (["BI-FAST CR%PT ECART WEBPORTAL%",
          "BI-FAST CR%FLIPTECH LENTERA I%",
          "BI-FAST CR%VISIONET INTERNASI%",
          "BI-FAST CR%NUSA SATU INTI ART%"], 
         "CR", "company"),
        
        (["SWITCHING%"], "CR", "individual"),
        (["KR OTOMATIS%PT%", "KR OTOMATIS%MANDIRI KREDIVO FINANCE%"], "CR", "company"),
        (["SETORAN%"], "CR", "individual"),
        (["KR INTERCHANGE%"], "CR", "individual"),
        (["CR KOREKSI%"], "CR", "individual"),
        (["BUNGA%"], "CR", "individual"),
        
        # === DB ===
        ([
            "%FTFVA/WS95031%",
            "%FTFVA/WS95271%",
            "%FTFVA/WS95051%",
            "%FTFVA/WS95011%",
            "%FTFVA/WS95221%",
            "%FTFVA/WS95261%",
            "%FTFVA/WS95071%"
        ], "DB", "company"),
    ],
    "MANDIRI": [],
    "BNI": [],
    "BRI": [],
    "SEABANK": []
}

def match_company(description):
    if description:
        desc_lower = description.lower()
        for name in company_list:
            if name in desc_lower:
                return name.upper()  # 或直接返回原始 name
    return None

match_company_udf = udf(match_company, StringType())

def build_comp_or_indiv_column(company_list, df, rules, result_col_name="comp_or_indiv", default_value="individual"):
    # 定义带 company_list 的 UDF


    # 先匹配公司词库
    df = df.withColumn("company_name_if_match", match_company_udf(col("description")))

    # 关键词模糊匹配
    condition = None
    for bank, pattern_list in rules.items():
        for patterns, cr_db, label in pattern_list:
            for pattern in patterns:
                regex = re.escape(pattern).replace(r"%", ".*")
                match_expr = (
                    (col("bank") == bank) &
                    (col("cr_db") == cr_db) &
                    col("description").rlike(regex)
                )
                condition = when(match_expr, label) if condition is None else condition.when(match_expr, label)

    # 优先使用公司词库匹配结果
    df = df.withColumn(
        result_col_name,
        when(col("company_name_if_match").isNotNull(), lit("company"))
        .otherwise(condition.otherwise(lit(default_value)))
    )

    return df.drop("company_name_if_match")

In [8]:
# sender pattern 配置字典

bank_sender_rules = {
    "BCA": [
        # (patterns, cr_db, sender)
        # CR
        (["TRSF E-BANKING CR%"], "CR", r"(?<!\w)([A-Z\s'-]+)\s*\d*$", 1),
        (["BI-FAST CR%"], "CR", r"%UPPER%(?<!\w)([A-Z\s'-]+)\s*\d*$", 1),
        (["KR OTOMATIS%AutoCr-PL%"], "CR", r"%UPPER%([A-Z\s'-]+)\s*\w*\s*AUTOCR-PL", 1),
        (["KR OTOMATIS%QR%", "KR OTOMATIS%TGH%"], "CR", r"%UPPER%([A-Z\s.\'-]+)\s*(QR|TGH)", 1),
        (["KR OTOMATIS%PT ECART WEBPORTAL%"], "CR", r"PT ECART WEBPORTAL", 0),
        (["KR OTOMATIS%MANDIRI KREDIVO FINANCE%"], "CR", r"MANDIRI KREDIVO FINANCE", 0),
        (["SETORAN%"], "CR", r"([A-Z\s'-]+)\s*\d*$", 1),
        (["SWITCHING%"], "CR", r"%UPPER%([A-Z\s.\'-, ]+)\s*/", 1),

        # DB
        (["TRSF E-BANKING DB%FTFVA%"], "DB",
         r"(?<!\w)([A-Za-z\s'.-]+)\s(- -)\s*\w*$", 1),

        (["BI-FAST DB BIF%", "SWITCHING DB%"], "DB",
         r"%UPPER%([A-Z\s\.\'-]+)\s*(MYBCA|M-BCA)", 1),

        (["TRANSAKSI DEBIT TGL%"], "DB",
         r".00(\.*)", 1),
    ],
    "MANDIRI": [],
    "BNI": [],
    "BRI": [],
    "SEABANK": []
}

def match_company(description):
    if description:
        desc_lower = description.lower()
        for name in company_list:
            if name in desc_lower:
                return name.upper()  # 或直接返回原始 name
    return None

match_company_udf = udf(match_company, StringType())

def build_sender_column(company_list, df, rules, result_col_name="sender"):
    df = df.withColumn("company_name_if_match", match_company_udf(col("description")))

    regex_condition = None
    for bank, pattern_list in rules.items():
        for rule in pattern_list:
            patterns, cr_db, regex_pattern, group = rule

            for pattern in patterns:
                like_expr = (col("bank") == bank) & (col("cr_db") == cr_db) & (col("description").like(pattern))

                target_col = col("description")
                if "%UPPER%" in regex_pattern:
                    regex_pattern = regex_pattern.replace("%UPPER%", "")
                    target_col = upper(col("description"))

                extract_expr = regexp_extract(target_col, regex_pattern, group)
                regex_condition = (
                    when(like_expr, extract_expr) if regex_condition is None else regex_condition.when(like_expr, extract_expr)
                )

    regex_condition = regex_condition.otherwise(lit(None))

    df = df.withColumn("regex_result", regex_condition)
    df = df.withColumn(result_col_name, F.coalesce(col("company_name_if_match"), col("regex_result")))

    return df.drop("company_name_if_match", "regex_result")

In [9]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

def build_company_type_column(df, company_dictionary_df, result_col_name="company_type"):
    """
    根据 description 匹配公司名称，直接从 company_dictionary_df 取 company_type
    :param df: 银行流水 DF
    :param company_dictionary_df: 公司字典 DF (company_name, company_type, remark)
    """
    # 用 lower 做统一大小写匹配
    df = df.withColumn("description_lower", F.lower(col("description")))
    company_dictionary_df = company_dictionary_df.withColumn("company_name_lower", F.lower(col("company_name")))
    
    # 模糊匹配 join
    joined_df = df.join(
        company_dictionary_df,
        df["description_lower"].contains(company_dictionary_df["company_name_lower"]),
        "left"
    )
    
    return joined_df.withColumnRenamed("company_type", result_col_name) \
                    .drop("description_lower", "company_name_lower")

In [11]:
from pyspark.sql.functions import col, upper, regexp_replace, when, lit, expr, split, length, substring

# 电商平台规则
ecommerce_platform_rules = {
    "BRI": [
        # BRI 特殊规则，写成函数式判断，或者放在外面单独写，或者简化成带参数的规则
        {
            "condition": lambda df: (
                (df.bank == "BRI") &
                (df.description.rlike(r"^\S+\s\S+")) &  # 至少两个词，简单示例
                (df.description.substr(0, 15) == df.acct_no) & 
                (length(split(df.description, " ")[0]) == 15) &
                (length(split(df.description, " ")[1]) == 16) &
                (substring(split(df.description, " ")[1], 1, 2) == substring(df.transaction_date, 3, 2)) &
                (
                    ((substring(df.transaction_date, 6, 2).cast("int") < 10) & 
                     (substring(split(df.description, " ")[1], 3, 1) == substring(df.transaction_date, 6, 2))) |
                    ((substring(df.transaction_date, 6, 2).cast("int") >= 10) & 
                     (substring(split(df.description, " ")[1], 3, 1) == expr("chr(ascii('A') + cast(substring(transaction_date, 6, 2) as int) - 10)")))
                )
            ),
            "label": "Grabfood_BRI"
        }
    ],
    "default": [
        # 非BRI银行通用规则
        {
            "pattern": r"VISIONETINTERNASI",
            "label": "Grabfood"
        },
        {
            "pattern": r"DOMPETANAKBANGSA.*WLST|WLST.*DOMPETANAKBANGSA",
            "label": "Gojek"
        },
        {
            "pattern": r"TOKOPEDIA",
            "label": "Tokopedia"
        },
        {
            "pattern": r"TIKTOK|TIK TOK|TTSIDL|TTS-IDL",
            "label": "Tiktok Shop"
        },
        {
            "pattern": r"ECART|LAZADA",
            "label": "Lazada"
        },
        {
            "pattern": r"GLOBALDIGITAL",
            "label": "Global Digital Niaga"
        },
        {
            "pattern": r"BUKALAPAKCOM",
            "label": "Bukalapak"
        },
    ]
}

def build_ecommerce_platform_column(df, rules):
    # 先处理BRI特殊规则
    bri_rules = rules.get("BRI", [])
    condition = None
    
    for rule in bri_rules:
        cond = rule["condition"](df)
        label = rule["label"]
        condition = when(cond, lit(label)) if condition is None else condition.when(cond, lit(label))

    # 处理默认通用规则
    default_rules = rules.get("default", [])
    for rule in default_rules:
        pattern = rule["pattern"]
        label = rule["label"]
        cond = regexp_replace(upper(col("description")), " ", "").rlike(pattern)
        condition = when(cond, lit(label)) if condition is None else condition.when(cond, lit(label))
    
    condition = condition.otherwise(lit(""))

    return df.withColumn("ecommerce_platform", condition)

In [19]:
# 主表
spark = SparkSession.builder.appName(f"ID Bank Pattern Extract")\
                            .getOrCreate()

main_tab_sql = 	f"select distinct acct_no, apply_no, bank_name as bank, bs_index, currency, credit_amt as amount , description, extra_infos, page_index, period_end_date, period_start_date, platform_user_id, remaining_balance, row_index, source_file, total_balance, total_credit_amt, total_credit_cnt, total_debit_amt, total_debit_cnt, transaction_date, transaction_type, user_address, user_name, biz_type, if(credit_amt>0, 'CR', 'DB') as cr_db from risk.dwd_extract_bankstatement_record_df where grass_date = '${ds}' and region = 'ID' " \

bank_record_df = spark.sql(main_tab_sql)


company_dictionary_sql = f"select distinct company_name, company_type, remark " \
                         f"from risk.dim_bank_record_company_dictionary " 

company_dictionary_df = spark.sql(company_dictionary_sql)
company_list = [row.company_name.lower() for row in company_dictionary_df.select("company_name").collect()]

In [13]:
# ETL
bank_record_df = (
    bank_record_df
    .transform(lambda df: build_txn_type_column(df, bank_txn_type_pattern_rules))
    .transform(lambda df: build_comp_or_indiv_column(company_list, df, bank_comp_or_indiv_pattern_rules))
    .transform(lambda df: build_sender_column(company_list, df, bank_sender_rules))
    .transform(lambda df: build_company_type_column(df, company_dictionary_df))
    .transform(lambda df: build_ecommerce_platform_column(df, ecommerce_platform_rules))
)

In [8]:
bank_record_df.createOrReplaceTempView("temp_view")
spark.sql(f"insert overwrite ${SCHEMA}.dwd_bank_record_id_category_df partition(grass_date) select distinct acct_no, apply_no, bank as bank_name, bs_index, currency, amount , description, extra_infos, page_index, period_end_date, period_start_date, platform_user_id, remaining_balance, row_index, source_file, total_balance, total_credit_amt, total_credit_cnt, total_debit_amt, total_debit_cnt, transaction_date, transaction_type, user_address, user_name, biz_type, txn_type, comp_or_indiv, sender, company_type, ecommerce_platform, cr_db, '${ds}' as grass_date from temp_view;")

DataFrame[]